# 🔬 Napari image inspection

A clean, step-by-step workflow for opening an OME-TIFF, reading its metadata, applying Fiji-style display ranges, and viewing channels in napari.

**Workflow:** ⚙️ setup → 📁 input → 📦 load → 📐 metadata → 🎚️ contrast → 🖥️ viewer

## ⚙️ 1. Prepare the environment

Set the GUI-related environment variables required by napari in this session.

In [1]:
import os
os.environ["DISPLAY"] = ":1"
os.environ["XDG_RUNTIME_DIR"] = "/tmp/runtime-jovyan"
os.environ["QT_QPA_PLATFORM"] = "xcb"

In [ ]:
import napari
import numpy as np
from bioio import BioImage

# Leave a channel out of this dict to auto-compute its contrast limits
# using Fiji's default algorithm (0.35% saturated pixels).
# To override, add its name here, e.g. {"DAPI": [100, 4000], "GFP": [50, 2000]}
CUSTOM_CONTRAST_LIMITS = {}
SATURATED_PIXELS_PCT = 0.35  # Fiji's default; increase for more contrast

## 📁 2. Choose the input image

Point the workflow to the OME-TIFF that should be inspected. Update `file_path` when working with another dataset.

In [ ]:
from pathlib import Path

# 📁 Choose the input image
file_path = Path("your_image_file.ome.tif")
input_path = file_path.expanduser().resolve()
print(f"Input image: {input_path}")

## 📦 3. Load the image data

Read the first timepoint in `CZYX` order so the channel axis is explicit for later display.

In [ ]:
img = BioImage(input_path)
data = img.get_image_data("CZYX", T=0)

print(f"Loaded data with shape CZYX = {data.shape}")

## 📐 4. Read physical scale and channel metadata

Extract voxel spacing, channel names, and OME-defined colors so napari can display the image in physical coordinates with meaningful labels.

In [ ]:
scale = img.physical_pixel_sizes
spatial_scale = [scale.Z, scale.Y, scale.X]

ome = img.ome_metadata
channels_meta = ome.images[0].pixels.channels
channel_names = [channel.name or f"Channel {index}" for index, channel in enumerate(channels_meta)]

colormaps = []
for channel in channels_meta:
    if channel.color is not None:
        red, green, blue = channel.color.as_rgb_tuple()[:3]
        colormaps.append([[0, 0, 0], [red / 255, green / 255, blue / 255]])
    else:
        colormaps.append("gray")

print(f"Channels: {', '.join(channel_names)}")
print(f"Spatial scale (Z, Y, X): {spatial_scale}")

## 🎚️ 5. Calculate display contrast

Use Fiji-style percentile clipping by default, while allowing selected channels to use custom limits through `CUSTOM_CONTRAST_LIMITS`.

In [ ]:
def fiji_auto_contrast(channel_data, saturated_pct=SATURATED_PIXELS_PCT):
    """Return Fiji-style display limits for one channel."""
    low_pct = saturated_pct / 2
    high_pct = 100 - saturated_pct / 2
    low, high = np.percentile(channel_data, [low_pct, high_pct])
    if high <= low:
        high = low + 1
    return [float(low), float(high)]


contrast_limits = [
    CUSTOM_CONTRAST_LIMITS.get(name, fiji_auto_contrast(channel_data))
    for name, channel_data in zip(channel_names, data)
]

print("CUSTOM_CONTRAST_LIMITS = {")
for name, limits in zip(channel_names, contrast_limits):
    print(f'    "{name}": [{limits[0]:.0f}, {limits[1]:.0f}],')
print("}")

## 🖥️ 6. Open the napari viewer

Create one viewer with additive channel blending, OME colors, physical scaling, and the calculated contrast limits.

In [ ]:
viewer = napari.Viewer()

viewer.add_image(
    data,
    channel_axis=0,
    scale=spatial_scale,
    name=channel_names,
    colormap=colormaps,
    contrast_limits=contrast_limits,
    blending="additive",
)